# Elections

This notebook cleans and combines mayoral, gubernatorial and presidential election results retrieved from the NYC Board of Elections website, then saves it to a CSV file.

## Import Libraries and Data

In [53]:
## import libraries
import pandas as pd
import geopandas as gpd
import csv

In [128]:
## import 2021 mayoral general election results
mayor_df = pd.read_csv('../input/elections/00000200000Citywide Mayor Citywide EDLevel.csv',
                       header=None,
                       dtype = {11:'object',
                      12:'object'})

In [129]:
## import 2022 gubernatorial election results
gov_df = pd.read_csv('../input/elections/00000200000Citywide Governor Lieutenant Governor Citywide EDLevel.csv',
                     header = None,
                     dtype = {11:'object',
                      12:'object'})

In [130]:
## read in the csv file that includes the 2024 presidential election data
prez_df = pd.read_csv("../input/elections/00000100000Citywide President Vice President Citywide EDLevel.csv",
                      header=None,
                      dtype = {11:'object',
                      12:'object'})

In [131]:
## read in the voter enrollment files
richmond = pd.read_excel("../input/voter_enrollment/richmonded_nov24.xlsx", dtype ={"ed": object})
bronx = pd.read_excel("../input/voter_enrollment/bronxed_nov24.xlsx", dtype ={"ed": object})
kings = pd.read_excel("../input/voter_enrollment/kingsed_nov24.xlsx", dtype ={"ed": object})
ny = pd.read_excel("../input/voter_enrollment/new-yorked_nov24.xlsx", dtype ={"ed": object})
queens = pd.read_excel("../input/voter_enrollment/queensed_nov24.xlsx", dtype ={"ed": object})

## Clean Results

In [114]:
## take a peak at the columns
prez_df.columns

Index([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19,
       20, 21],
      dtype='int64')

In [115]:
gov_df.columns

Index([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19,
       20, 21],
      dtype='int64')

In [132]:
mayor_df.columns

Index([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19,
       20, 21],
      dtype='int64')

In [133]:
## drop the first few columns, which are just the headers repeated over and over again
prez_df = prez_df.drop([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10], axis = 1)
gov_df = gov_df.drop([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10], axis = 1)
mayor_df = mayor_df.drop([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10], axis = 1)

In [134]:
## create a list of new headers
headers = ['AD', 'ED', 'County', 'EDAD Status', 'Event', 'Party/Independent Body',
       'Office/Position Title', 'District Key', 'VoteFor', 'Unit Name',
       'Tally']

In [135]:
## assign those headers while changing them all to lowercase
prez_df.columns = headers
prez_df.columns = prez_df.columns.str.lower()

## do it for gov df too
gov_df.columns = headers
gov_df.columns = gov_df.columns.str.lower()

## and mayor df
mayor_df.columns = headers
mayor_df.columns = mayor_df.columns.str.lower()

In [137]:
## create new column that combines ad and ed
prez_df['ad_ed'] = prez_df['ad'] + prez_df['ed']
gov_df['ad_ed'] = gov_df['ad'] + gov_df['ed']
mayor_df['ad_ed'] = mayor_df['ad'] + mayor_df['ed']

In [83]:
## change tally column to integer dtype
prez_df["tally"] = (pd.to_numeric(prez_df["tally"].astype(str).str.replace(",", "", regex=False),
                                  errors="coerce"))

In [122]:
## change tally column to integer dtype
gov_df["tally"] = (pd.to_numeric(gov_df["tally"].astype(str).str.replace(",", "", regex=False),
                                  errors="coerce"))

In [138]:
## change tally column to integer dtype
mayor_df["tally"] = (pd.to_numeric(mayor_df["tally"].astype(str).str.replace(",", "", regex=False),
                                  errors="coerce"))

In [ ]:
## pivot president df
prez_pivot = pd.pivot_table(prez_df,
                            index='ad_ed',
                            values='tally',
                            columns='unit name',
                            aggfunc='sum')

In [124]:
## pivot president df
gov_pivot = pd.pivot_table(gov_df,
                            index='ad_ed',
                            values='tally',
                            columns='unit name',
                            aggfunc='sum')

In [139]:
## pivot president df
mayor_pivot = pd.pivot_table(mayor_df,
                            index='ad_ed',
                            values='tally',
                            columns='unit name',
                            aggfunc='sum')

In [142]:
## reset indexes
prez_pivot = prez_pivot.reset_index()
gov_pivot = gov_pivot.reset_index()
mayor_pivot = mayor_pivot.reset_index()

ValueError: cannot insert level_0, already exists

In [127]:
gov_pivot = gov_pivot.rename(columns={'Absentee / Military':'gov_absentee',
                                      'Affidavit':'gov_affidavit',
                                      'Federal':'gov_federal',
                                      'Kathy C. Hochul / Antonio Delgado (Democratic)':'hochul_dem',
                                      'Kathy C. Hochul / Antonio Delgado (Working Families)':'hochul_wfp',
                                      'Lee Zeldin / Alison Esposito (Conservative)':'zeldin_con',
                                      'Lee Zeldin / Alison Esposito (Republican)':'zeldin_rep',
                                      'Manually Counted Emergency':'manual_count',
                                      'Public Counter':'public_count',
                                      'Scattered':'scattered'})

In [96]:
prez_pivot = prez_pivot.rename(columns={'Absentee / Military':'absentee',
                                        'Affidavit':'affidavit',
                                        'Donald J. Trump / JD Vance (Conservative)':'trump_conservative',
                                        'Donald J. Trump / JD Vance (Republican)':'trump_rep',
                                        'Federal':'federal',
                                        'Kamala D. Harris / Tim Walz (Democratic)':'harris_dem',
                                        'Kamala D. Harris / Tim Walz (Working Families)':'harris_wfp',
                                        'Manually Counted Emergency':'manual_count',
                                        'Public Counter':'public_counter',
                                        'Scattered':'scattered'})

In [143]:
mayor_pivot = mayor_pivot.rename(columns={'Absentee / Military':'absentee',
                                          'Affidavit':'affidavit',
                                          'Catherine Rojas (Socialism & Lib)':'rojas',
                                          'Curtis A. Sliwa (Independent)':'sliwa_ind',
                                          'Curtis A. Sliwa (Republican)':'sliwa_rep',
                                          'Eric L. Adams (Democratic)':'adams_dem',
                                          'Fernando Mateo (Save Our City)':'mateo',
                                          'Manually Counted Emergency':'manual_count',
                                          'Public Counter':'public_counter',
                                          'Quanda S. Francis (Empowerment)':'francis',
                                          'Raja Michael Flores (Humanity United)':'flores',
                                          'Scattered':'scattered',
                                          'Skiboky Stora (Out Lawbreaker)':'stora',
                                          'Stacey H. Prussman (Libertarian)':'prussman',
                                          'William A. Pepitone (Conservative)':'pepitone'})

## Create new columns for total votes

In [158]:
## create total vote counts for overall, harris and trump
prez_pivot['harris'] = prez_pivot['harris_dem'] + prez_pivot['harris_wfp']
prez_pivot['trump'] = prez_pivot['trump_conservative'] + prez_pivot['trump_rep']
prez_pivot['prez_tot_votes'] = prez_pivot['public_counter'] + prez_pivot['manual_count'] + prez_pivot['absentee'] + prez_pivot['affidavit']


In [160]:
gov_pivot['zeldin'] = gov_pivot['zeldin_con'] + gov_pivot['zeldin_rep']
gov_pivot['hochul'] = gov_pivot['hochul_dem'] + gov_pivot['hochul_wfp']
gov_pivot['tot_gov_votes'] = gov_pivot['gov_affidavit'] + gov_pivot['gov_absentee'] + gov_pivot['public_count'] + gov_pivot['manual_count']

In [165]:
mayor_pivot['sliwa'] = mayor_pivot['sliwa_rep'] + mayor_pivot['sliwa_ind']
mayor_pivot['tot_mayor_votes'] = mayor_pivot['public_counter'] + mayor_pivot['manual_count'] + mayor_pivot['affidavit'] + mayor_pivot['absentee']
mayor_pivot = mayor_pivot.rename(columns={'adams_dem':'adams'})

In [167]:
mayor_cleaned = mayor_pivot.loc[:,['ad_ed',
                                   'sliwa',
                                   'adams',
                                   'francis',
                                   'flores',
                                   'stora',
                                   'prussman',
                                   'pepitone',
                                   'tot_mayor_votes']]

In [161]:
gov_cleaned = gov_pivot.loc[:,['ad_ed',
                               'zeldin',
                               'hochul',
                               'tot_gov_votes']]

In [156]:
prez_cleaned = prez_pivot.loc[:,['ad_ed',
                                 'harris',
                                 'trump',
                                 'prez_tot_votes']]

## Combine

In [170]:
first_combo = pd.merge(prez_cleaned,
                       mayor_cleaned,
                       on='ad_ed',
                       how='left')

In [171]:
combined_df = pd.merge(first_combo,
                       gov_cleaned,
                       on='ad_ed',
                       how='left')

In [172]:
combined_df

unit name,ad_ed,harris,trump,prez_tot_votes,sliwa,adams,francis,flores,stora,prussman,pepitone,tot_mayor_votes,zeldin,hochul,tot_gov_votes
0,23001,295.0,1011.0,1325.0,469.0,107.0,1.0,0.0,0.0,2.0,49.0,639.0,918.0,213.0,1139.0
1,23002,289.0,957.0,1260.0,499.0,106.0,0.0,0.0,0.0,1.0,56.0,682.0,877.0,186.0,1069.0
2,23003,99.0,330.0,433.0,475.0,78.0,0.0,1.0,0.0,1.0,41.0,603.0,252.0,74.0,329.0
3,23004,391.0,871.0,1276.0,66.0,27.0,0.0,0.0,0.0,0.0,3.0,103.0,738.0,288.0,1030.0
4,23005,336.0,936.0,1289.0,311.0,135.0,0.0,0.0,0.0,0.0,8.0,468.0,829.0,250.0,1085.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4414,87055,485.0,186.0,687.0,23.0,115.0,2.0,0.0,0.0,1.0,0.0,145.0,71.0,261.0,335.0
4415,87056,5.0,3.0,8.0,22.0,99.0,0.0,0.0,0.0,0.0,1.0,129.0,0.0,2.0,2.0
4416,87057,0.0,0.0,0.0,29.0,154.0,0.0,0.0,0.0,1.0,1.0,191.0,NaN,NaN,0.0
4417,87058,NaN,NaN,0.0,11.0,112.0,0.0,0.0,0.0,0.0,4.0,135.0,0.0,0.0,0.0


In [173]:
combined_df.to_csv('../output/elections/elections.csv')